# 02 — Phase-1 Baseline Training — **YOLOPv2 best-row target**

**Primary target.** This notebook's default `CONFIG = 'YOLOPv2-best-row'`
reproduces the **final cumulative ablation row** from YOLOPv2 §3:
ELAN/group-conv backbone + SPP neck + FPN+PAN + Mosaic + MixUp + focal
loss on cls/obj + focal loss on lane seg + **hybrid focal+dice on lane
seg**. Drivable-area head is the only intentional deviation.

**Available configs** (flip `CONFIG` in cell 4):

| CONFIG | YAML | what it targets |
|---|---|---|
| `'YOLOPv2-best-row'`   | `configs/yolopv2_best_row.yaml`             | best row — focal + **dice** + cosine warmup |
| `'YOLOPv2-focal-only'` | `configs/yolopv2_focal_only_ablation.yaml`  | the row right before best (focal only) |
| `'YOLOP'`              | `configs/yolop_vehicle_lane_baseline.yaml`  | honest YOLOP minus DA (reference) |

Every delta vs YOLOP upstream is documented in
`AUDIT_YOLOPV2_ALIGNMENT.md` and `FINAL_ALIGNMENT_STATUS_v4.md`.
Values that the YOLOPv2 public release does not publish (focal γ,
dice weight, SGDR T_0/T_mult, E-ELAN `groups`) are marked `[INFERRED]`
in the YAML and in code.

**Output** (per config): `checkpoints/<run>/{latest,best}.pth`,
`tb_logs/<run>/`, `metrics/<run>/` — all under
`/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/`.

**Validation letterbox:** YOLOPv2 paper uses **train 640×640, test
640×384**. Cell 6 passes an explicit `(H, W)` tuple to `BddDataset`
so the val loader uses `letterbox(auto=False)` and the output shape
is exactly 384×640, not stride-rounded by accident.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yacs tqdm opencv-python-headless tensorboard

Mounted at /content/drive


In [2]:
import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [3]:
import torch
import torch.nn as nn
import numpy as np
import logging
import math
from pathlib import Path
from torch import amp  # torch.amp replaces torch.cuda.amp in recent PyTorch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, train, validate
from lib.dataset import BddDataset

# Run mode — stage1 has two sanctioned modes:
#   'smoke' : 16-sample subset, 2 epochs, val plots off. For fast debug.
#   'full'  : full BDD100K, cfg.TRAIN.END_EPOCH epochs, val plots at epoch 0 only.
# The selected mode is applied at dataset-build + train-loop time; nothing
# about the model / loss / optimizer is changed between modes.
RUN_MODE = 'full'    # 'smoke' | 'full'

# Speed knobs — safe across CUDA versions, do not change baseline semantics.
# - TF32: ~2x matmul throughput on A5000/A100 with negligible accuracy loss.
# - cudnn.benchmark: picks the fastest conv algorithm for fixed-shape batches
#   (we letterbox to a fixed HxW per-split, so this is safe).
# - channels_last is an OPTIONAL toggle because small spatial heads in this
#   model (lane stride-1 decoder) see uneven speedup; leave it off by default.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'RUN_MODE = {RUN_MODE}')

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
RUN_MODE = full


In [4]:

# ── Config selection + dataset roots ──
# Stage1 primary baseline = YOLOP. Flip CONFIG manually for YOLOPv2 rows.
CONFIG = 'YOLOP'   # 'YOLOP' | 'YOLOPv2-paper-no-da' | 'YOLOPv2-best-row' | 'YOLOPv2-focal-only'
GPU_PROFILE = 'G4_RTX_PRO_6000'   # 'none' | 'G4_RTX_PRO_6000'
BATCH_OVERRIDE = None             # e.g. 24 to force a run; None keeps YAML

from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

yaml_map = {
    'YOLOPv2-paper-no-da': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_paper_no_da.yaml'),
    'YOLOPv2-best-row':   os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_best_row.yaml'),
    'YOLOPv2-focal-only': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_focal_only_ablation.yaml'),
    'YOLOP':              os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolop_vehicle_lane_baseline.yaml'),
}
run_name = {
    'YOLOPv2-paper-no-da': 'yolopv2_paper_no_da',
    'YOLOPv2-best-row':   'yolopv2_best_row',
    'YOLOPv2-focal-only': 'yolopv2_focal_only',
    'YOLOP':              'yolop',
}[CONFIG]

cfg.defrost()
cfg.merge_from_file(yaml_map[CONFIG])

# Resolve dataset roots (Colab SSD -> Drive fallback).
ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)

cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')

cfg.DRIVE.ROOT = ECOCAR_ROOT
cfg.DRIVE.CHECKPOINT_DIR = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'checkpoints', run_name)
cfg.DRIVE.METRICS_DIR    = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'metrics',     run_name)

# G4 / RTX PRO 6000 stability profile.
if BATCH_OVERRIDE is not None:
    cfg.TRAIN.BATCH_SIZE_PER_GPU = int(BATCH_OVERRIDE)

if GPU_PROFILE == 'G4_RTX_PRO_6000':
    cfg.TRAIN.GRAD_CLIP_NORM = 10.0
    cfg.WORKERS = min(int(cfg.WORKERS), 4)

    if CONFIG == 'YOLOP':
        # YOLOP upstream uses Adam lr=1e-3 at batch 24. On Colab G4 with AMP,
        # a slightly softer LR / longer warmup stabilizes the lane branch.
        if cfg.TRAIN.BATCH_SIZE_PER_GPU >= 24:
            cfg.TRAIN.LR0 = 8e-4
        cfg.TRAIN.WARMUP_EPOCHS = max(float(cfg.TRAIN.WARMUP_EPOCHS), 5.0)
        cfg.LOSS.LL_IOU_GAIN = min(float(cfg.LOSS.LL_IOU_GAIN), 0.1)
    else:
        # YOLOPv2 rows are more augmentation-heavy; on G4 they benefit from
        # a longer warmup and slightly gentler mosaic/mixup probabilities.
        if cfg.TRAIN.BATCH_SIZE_PER_GPU >= 24:
            cfg.TRAIN.LR0 = 0.0075
        else:
            cfg.TRAIN.LR0 = min(float(cfg.TRAIN.LR0), 0.005)
        cfg.TRAIN.WARMUP_EPOCHS = max(float(cfg.TRAIN.WARMUP_EPOCHS), 5.0)
        cfg.DATASET.MOSAIC_PROB = min(float(getattr(cfg.DATASET, 'MOSAIC_PROB', 1.0)), 0.7)
        cfg.DATASET.MIXUP_PROB = min(float(getattr(cfg.DATASET, 'MIXUP_PROB', 0.15)), 0.10)

cfg.TEST.PLOTS = False
if RUN_MODE == 'smoke':
    cfg.TRAIN.END_EPOCH = 2
    cfg.TRAIN.VAL_FREQ = 1
    cfg.TEST.PLOTS = True

cfg.freeze()

os.makedirs(cfg.DRIVE.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.DRIVE.METRICS_DIR, exist_ok=True)
output_dir = cfg.DRIVE.METRICS_DIR
tb_log_dir = os.path.join(cfg.DRIVE.ROOT, 'yolop_vehicle_lane', 'stage1', 'tb_logs', run_name)
os.makedirs(tb_log_dir, exist_ok=True)

print(f'Config       : {CONFIG}')
print(f'YAML         : {yaml_map[CONFIG]}')
print(f'GPU profile  : {GPU_PROFILE}')
print(f'Model.NAME   : {cfg.MODEL.NAME}')
print(f'NC           : {cfg.MODEL.NC}  (classes: {cfg.MODEL.VEHICLE_CLASSES})')
print(f'Optimizer    : {cfg.TRAIN.OPTIMIZER}  LR0={cfg.TRAIN.LR0}  WD={cfg.TRAIN.WD}')
print(f'Batch(train) : {cfg.TRAIN.BATCH_SIZE_PER_GPU}   Batch(val): {cfg.TEST.BATCH_SIZE_PER_GPU}')
print(f'Warmup       : {cfg.TRAIN.WARMUP_EPOCHS}   GradClip: {getattr(cfg.TRAIN, "GRAD_CLIP_NORM", 0.0)}')
print(f'Focal γ      : cls/obj={cfg.LOSS.FL_GAMMA}  lane={getattr(cfg.LOSS, "LL_FL_GAMMA", 0.0)}')
print(f'Dice gain    : {getattr(cfg.LOSS, "LL_DICE_GAIN", 0.0)}   LL_IoU gain: {cfg.LOSS.LL_IOU_GAIN}')
print(f'Mosaic/MixUp : {getattr(cfg.DATASET, "MOSAIC", False)} / {getattr(cfg.DATASET, "MIXUP", False)}')
print(f'Val plots    : {cfg.TEST.PLOTS}')
print(f'Checkpoints  : {cfg.DRIVE.CHECKPOINT_DIR}')


Extracting /content/drive/MyDrive/EcoCAR/datasets/bdd100k_vehicle5.tar.gz into this notebook runtime ...
Config       : YOLOP
YAML         : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/configs/yolop_vehicle_lane_baseline.yaml
GPU profile  : G4_RTX_PRO_6000
Model.NAME   : YOLOP
NC           : 1  (classes: ['vehicle'])
Optimizer    : adam  LR0=0.001  WD=0.0005
Batch(train) : 16   Batch(val): 24
Warmup       : 5.0   GradClip: 10.0
Focal γ      : cls/obj=0.0  lane=0.0
Dice gain    : 0.0   LL_IoU gain: 0.1
Mosaic/MixUp : False / False
Val plots    : False
Checkpoints  : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolop


In [5]:
# ── Logger ──
logger = logging.getLogger('train')
logger.setLevel(logging.INFO)
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)

In [6]:
# ── Build datasets ──
# YOLOPv2 paper §3: train 640×640 letterbox, test 640×384 letterbox.
# MODEL.IMAGE_SIZE is written (W, H) in the YAML. BddDataset accepts
# either an int (square auto-letterbox) or an explicit (H, W) tuple
# (rectangular letterbox with auto=False).
#
# Long-run hygiene (REPAIR v5):
#   * val plots are expensive and are the biggest non-science cost once
#     training is working. We enable them for epoch 0 only as an alignment
#     snapshot. If the user wants every epoch's plots, they can flip
#     `cfg.TEST.PLOTS = True` between val calls.
#   * persistent workers + prefetch keep the DataLoader warm between
#     epochs — otherwise every epoch pays the worker startup cost.
transform = T.ToTensor()

train_wh = tuple(cfg.MODEL.IMAGE_SIZE)              # (W, H)
val_wh = tuple(getattr(cfg.TEST, 'IMAGE_SIZE', cfg.MODEL.IMAGE_SIZE))

train_size = int(max(train_wh))                     # 640 — square aug-friendly
val_size = (int(val_wh[1]), int(val_wh[0]))         # (H, W) for letterbox auto=False

train_dataset = BddDataset(cfg, is_train=True,  inputsize=train_size, transform=transform)
val_dataset   = BddDataset(cfg, is_train=False, inputsize=val_size,   transform=transform)

# Smoke mode applies a fixed-seed tiny subset so we can detect real learning
# signal in 1-2 minutes instead of burning a GPU hour. The SAME samples are
# used every iteration so the overfit-a-tiny-subset check is meaningful.
if RUN_MODE == 'smoke':
    import random as _rnd
    _g = _rnd.Random(1234)
    idx = sorted(_g.sample(range(len(train_dataset)), 16))
    train_dataset.db = [train_dataset.db[i] for i in idx]
    idx_v = sorted(_g.sample(range(len(val_dataset)), 16))
    val_dataset.db = [val_dataset.db[i] for i in idx_v]
    print(f'[SMOKE] reduced train→{len(train_dataset)} val→{len(val_dataset)}')

train_loader = DataLoader(
    train_dataset, batch_size=cfg.TRAIN.BATCH_SIZE_PER_GPU,
    shuffle=cfg.TRAIN.SHUFFLE, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=train_dataset.collate_fn,
    persistent_workers=(cfg.WORKERS > 0),
    prefetch_factor=2 if cfg.WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
    shuffle=False, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=val_dataset.collate_fn,
    persistent_workers=(cfg.WORKERS > 0),
    prefetch_factor=2 if cfg.WORKERS > 0 else None,
)

# Sanity check — one sample tells us the actual letterboxed HxW used at train vs eval.
sample_train = train_dataset[0][0].shape
sample_val = val_dataset[0][0].shape
print(f'Train: {len(train_dataset)} samples, letterbox C×H×W = {tuple(sample_train)}')
print(f'Val:   {len(val_dataset)} samples, letterbox C×H×W = {tuple(sample_val)}')

[Dataset] split=train | layout=explicit_packaged_like
[Dataset] images=/content/bdd100k_raw/100k/train
[Dataset] labels=/content/bdd100k_raw/100k/train
[Dataset] lanes =/content/bdd100k_vehicle5/masks/train
building database...


100%|██████████| 70000/70000 [00:06<00:00, 10340.73it/s]


database build finish: 70000 samples
missing lane masks skipped: 0
missing detection labels: 0
[Dataset] split=val | layout=explicit_packaged_like
[Dataset] images=/content/bdd100k_raw/100k/val
[Dataset] labels=/content/bdd100k_raw/100k/val
[Dataset] lanes =/content/bdd100k_vehicle5/masks/val
building database...


100%|██████████| 10000/10000 [00:00<00:00, 10405.37it/s]


database build finish: 10000 samples
missing lane masks skipped: 0
missing detection labels: 0
Train: 70000 samples, letterbox C×H×W = (3, 384, 640)
Val:   10000 samples, letterbox C×H×W = (3, 640, 640)


In [7]:
# ── Build model, loss, optimizer, scheduler ──
# Scheduler policy
# ----------------
# YOLOPv2 paper §3 describes "cosine annealing with warm restart". Two
# interpretations are compatible with that phrasing:
#   (A) Per-iteration **linear warmup** for the first `WARMUP_EPOCHS`
#       epochs (the "warm restart" = warming up from zero), followed by
#       per-epoch **cosine annealing** from LR0 to LR0 * LRF over
#       the remaining epochs.
#   (B) True **SGDR** (CosineAnnealingWarmRestarts) with periodic
#       restarts throughout training.
#
# The paper does not publish `T_0` / `T_mult` or any restart period, so
# the SGDR variant would require inventing those. We default to (A) —
# which is YOLOP's own scheduler, matches the plain-English paper
# phrasing, and is well-calibrated to the 300-epoch budget — and expose
# `TRAIN.SGDR` + `TRAIN.SGDR_T0` knobs so a future ablation can swap in
# (B) without touching model / loss code. Marked [INFERRED] either way.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# get_net reads cfg.MODEL.NC and builds Detect with the right class count
# and sets model.names from the class-protocol registry. Do NOT override
# them here — doing so breaks the detector head / NMS label indexing
# (that was the KeyError bug in the stage1-1c run).
model = get_net(cfg).to(device)
model.gr = 1.0

criterion = get_loss(cfg, device)

if cfg.TRAIN.OPTIMIZER == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.TRAIN.LR0,
                                 betas=(cfg.TRAIN.MOMENTUM, 0.999))
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.TRAIN.LR0,
                                momentum=cfg.TRAIN.MOMENTUM,
                                weight_decay=cfg.TRAIN.WD,
                                nesterov=cfg.TRAIN.NESTEROV)
for pg in optimizer.param_groups:
    pg['initial_lr'] = cfg.TRAIN.LR0

use_sgdr = bool(getattr(cfg.TRAIN, 'SGDR', False))
if use_sgdr:
    sgdr_t0 = int(getattr(cfg.TRAIN, 'SGDR_T0', max(1, cfg.TRAIN.END_EPOCH // 3)))
    sgdr_tmult = int(getattr(cfg.TRAIN, 'SGDR_TMULT', 1))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=sgdr_t0, T_mult=sgdr_tmult,
        eta_min=cfg.TRAIN.LR0 * cfg.TRAIN.LRF,
    )
    sched_desc = f'SGDR (T_0={sgdr_t0}, T_mult={sgdr_tmult})'
else:
    lf = lambda x: ((1 + math.cos(x * math.pi / cfg.TRAIN.END_EPOCH)) / 2) * \
                   (1 - cfg.TRAIN.LRF) + cfg.TRAIN.LRF
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)
    sched_desc = 'cosine-annealing + linear warmup (YOLOP default)'

scaler = amp.GradScaler(device.type, enabled=device.type != 'cpu')

num_params = sum(p.numel() for p in model.parameters())
print(f'Model nc/names: {model.nc}  {model.names}')
print(f'Model params : {num_params/1e6:.2f}M')
print(f'Device       : {device}')
print(f'Optimizer    : {cfg.TRAIN.OPTIMIZER} @ LR0={cfg.TRAIN.LR0}, WD={cfg.TRAIN.WD}')
print(f'Scheduler    : {sched_desc}')
print(f'Warmup       : {cfg.TRAIN.WARMUP_EPOCHS} epochs (iter-based linear in-loop)')

Model nc/names: 1  ['vehicle']
Model params : 7.60M
Device       : cuda
Optimizer    : adam @ LR0=0.001, WD=0.0005
Scheduler    : SGDR (T_0=100, T_mult=1)
Warmup       : 5.0 epochs (iter-based linear in-loop)


In [8]:
# ── Resume from checkpoint if available ──
start_epoch = cfg.TRAIN.BEGIN_EPOCH
best_map = 0.0
best_ll_iou = 0.0

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(ckpt_path) and cfg.AUTO_RESUME:
    ckpt = torch.load(ckpt_path, map_location=device)
    try:
        model.load_state_dict(ckpt['state_dict'])
    except RuntimeError as e:
        # REPAIR (v5): old checkpoints saved with the 5-class detect head
        # are incompatible with the now-correct 1-class stage1 head. Bail
        # out with a clear message rather than silently dropping weights.
        print('[warn] resume failed due to shape mismatch:', e)
        print('       Delete old latest.pth and retrain from scratch.')
        raise
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_map = ckpt.get('best_map', 0.0)
    best_ll_iou = ckpt.get('best_ll_iou', 0.0)
    print(f'Resumed from epoch {start_epoch}, best_map={best_map:.4f}, best_ll_iou={best_ll_iou:.4f}')
else:
    print('Training from scratch')

Training from scratch


In [ ]:
# ── Training loop ──
# Checkpoint-selection policy (REPAIR v5):
#   * best_det.pth    — best mAP50
#   * best_lane.pth   — best lane IoU
#   * best_joint.pth  — best joint score  (also mirrored as `best.pth`)
# Joint score: `0.5 * mAP50 + 0.5 * lane_IoU`. Equal-weighted because both
# are already in [0,1] and stage1 should not prefer one task.
writer = SummaryWriter(tb_log_dir)
writer_dict = {
    'writer': writer,
    'train_global_steps': start_epoch * len(train_loader),
}

num_batch = len(train_loader)
num_warmup = max(round(cfg.TRAIN.WARMUP_EPOCHS * num_batch), 1000)

best_joint = 0.0

def _save_ckpt(name: str, state: dict):
    path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, name)
    torch.save(state, path)
    return path

def _set_plots(enabled: bool):
    """Toggle cfg.TEST.PLOTS on the frozen CfgNode."""
    cfg.defrost()
    cfg.TEST.PLOTS = bool(enabled)
    cfg.freeze()

# cfg.TRAIN.END_EPOCH
for epoch in range(start_epoch, cfg.TRAIN.END_EPOCH):
    train(cfg, train_loader, model, criterion, optimizer, scaler,
          epoch, num_batch, num_warmup, writer_dict, logger, device)

    scheduler.step()

    if epoch % cfg.TRAIN.VAL_FREQ == 0:
        # Plot only at epoch 0 for full runs (alignment snapshot); smoke
        # mode keeps plots on for every epoch so the user can eyeball
        # overfit behavior on the tiny subset.
        if RUN_MODE == 'full':
            _set_plots(epoch == 0)

        ll_seg_result, det_result, val_loss, maps, times = validate(
            epoch, cfg, val_loader, val_dataset, model, criterion,
            output_dir, tb_log_dir, writer_dict, logger, device
        )

        ll_acc, ll_iou, ll_miou = ll_seg_result
        mp, mr, map50, map_all = det_result
        joint = 0.5 * float(map50) + 0.5 * float(ll_iou)

        writer.add_scalar('val/loss', val_loss, epoch)
        writer.add_scalar('val/mAP50', map50, epoch)
        writer.add_scalar('val/mAP', map_all, epoch)
        writer.add_scalar('val/ll_iou', ll_iou, epoch)
        writer.add_scalar('val/ll_acc', ll_acc, epoch)
        writer.add_scalar('val/joint_score', joint, epoch)

        logger.info(
            f'Epoch {epoch} | mAP50={map50:.4f} mAP={map_all:.4f} | '
            f'LL_IoU={ll_iou:.4f} LL_Acc={ll_acc:.4f} | '
            f'Joint={joint:.4f} | Loss={val_loss:.4f}'
        )

        state = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_map': best_map,
            'best_ll_iou': best_ll_iou,
            'best_joint': best_joint,
            'joint_score': joint,
            'mAP50': float(map50),
            'll_iou': float(ll_iou),
        }
        _save_ckpt('latest.pth', state)

        if map50 > best_map:
            best_map = float(map50)
            _save_ckpt('best_det.pth', state)
            logger.info(f'  ** new best_det: mAP50={best_map:.4f}')
        if ll_iou > best_ll_iou:
            best_ll_iou = float(ll_iou)
            _save_ckpt('best_lane.pth', state)
            logger.info(f'  ** new best_lane: LL_IoU={best_ll_iou:.4f}')
        if joint > best_joint:
            best_joint = joint
            _save_ckpt('best_joint.pth', state)
            logger.info(f'  ** new best_joint: {best_joint:.4f}')
            # Historic alias — stage2/3 notebooks still load 'best.pth'.
            _save_ckpt('best.pth', state)

writer.close()
print(f'\nTraining complete. best_det mAP50={best_map:.4f} | '
      f'best_lane LL_IoU={best_ll_iou:.4f} | best_joint={best_joint:.4f}')


Epoch: [0][0/4375]	Time 14.176s (14.176s)	Speed 1.1 samples/s	Data 0.707s (0.707s)	Loss 0.43375 (0.43375)
INFO:train:Epoch: [0][0/4375]	Time 14.176s (14.176s)	Speed 1.1 samples/s	Data 0.707s (0.707s)	Loss 0.43375 (0.43375)
Epoch: [0][20/4375]	Time 0.058s (0.729s)	Speed 274.5 samples/s	Data 0.000s (0.034s)	Loss 0.43152 (0.43379)
INFO:train:Epoch: [0][20/4375]	Time 0.058s (0.729s)	Speed 274.5 samples/s	Data 0.000s (0.034s)	Loss 0.43152 (0.43379)
Epoch: [0][40/4375]	Time 0.052s (0.401s)	Speed 306.0 samples/s	Data 0.000s (0.018s)	Loss 0.42941 (0.43316)
INFO:train:Epoch: [0][40/4375]	Time 0.052s (0.401s)	Speed 306.0 samples/s	Data 0.000s (0.018s)	Loss 0.42941 (0.43316)
Epoch: [0][60/4375]	Time 0.052s (0.288s)	Speed 310.2 samples/s	Data 0.000s (0.012s)	Loss 0.43176 (0.43268)
INFO:train:Epoch: [0][60/4375]	Time 0.052s (0.288s)	Speed 310.2 samples/s	Data 0.000s (0.012s)	Loss 0.43176 (0.43268)
Epoch: [0][80/4375]	Time 0.053s (0.230s)	Speed 303.7 samples/s	Data 0.000s (0.009s)	Loss 0.43143 (0.43

                 all       1e+04    1.08e+05       0.026       0.719       0.338       0.106
Speed: 1.3/0.8/2.1 ms inference/NMS/total per 640x640 image at batch-size 16


Epoch 0 | mAP50=0.3382 mAP=0.1060 | LL_IoU=0.0557 LL_Acc=0.3311 | Joint=0.1970 | Loss=0.3009
INFO:train:Epoch 0 | mAP50=0.3382 mAP=0.1060 | LL_IoU=0.0557 LL_Acc=0.3311 | Joint=0.1970 | Loss=0.3009


Results saved to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolop/visualization


  ** new best_det: mAP50=0.3382
INFO:train:  ** new best_det: mAP50=0.3382
  ** new best_lane: LL_IoU=0.0557
INFO:train:  ** new best_lane: LL_IoU=0.0557
  ** new best_joint: 0.1970
INFO:train:  ** new best_joint: 0.1970
Epoch: [1][0/4375]	Time 0.258s (0.258s)	Speed 61.9 samples/s	Data 0.180s (0.180s)	Loss 0.32462 (0.32462)
INFO:train:Epoch: [1][0/4375]	Time 0.258s (0.258s)	Speed 61.9 samples/s	Data 0.180s (0.180s)	Loss 0.32462 (0.32462)
Epoch: [1][20/4375]	Time 0.052s (0.065s)	Speed 309.6 samples/s	Data 0.000s (0.009s)	Loss 0.31019 (0.31801)
INFO:train:Epoch: [1][20/4375]	Time 0.052s (0.065s)	Speed 309.6 samples/s	Data 0.000s (0.009s)	Loss 0.31019 (0.31801)
Epoch: [1][40/4375]	Time 0.051s (0.060s)	Speed 311.7 samples/s	Data 0.000s (0.005s)	Loss 0.31979 (0.31893)
INFO:train:Epoch: [1][40/4375]	Time 0.051s (0.060s)	Speed 311.7 samples/s	Data 0.000s (0.005s)	Loss 0.31979 (0.31893)
Epoch: [1][60/4375]	Time 0.052s (0.058s)	Speed 308.8 samples/s	Data 0.000s (0.003s)	Loss 0.32673 (0.31802)
I

...

  ** new best_joint: 0.4486
INFO:train:  ** new best_joint: 0.4486
Epoch: [16][0/4375]	Time 0.238s (0.238s)	Speed 67.1 samples/s	Data 0.171s (0.171s)	Loss 0.17522 (0.17522)
INFO:train:Epoch: [16][0/4375]	Time 0.238s (0.238s)	Speed 67.1 samples/s	Data 0.171s (0.171s)	Loss 0.17522 (0.17522)
Epoch: [16][20/4375]	Time 0.057s (0.065s)	Speed 280.7 samples/s	Data 0.000s (0.008s)	Loss 0.17804 (0.18228)
INFO:train:Epoch: [16][20/4375]	Time 0.057s (0.065s)	Speed 280.7 samples/s	Data 0.000s (0.008s)	Loss 0.17804 (0.18228)
Epoch: [16][40/4375]	Time 0.057s (0.060s)	Speed 280.9 samples/s	Data 0.000s (0.004s)	Loss 0.17408 (0.18244)
INFO:train:Epoch: [16][40/4375]	Time 0.057s (0.060s)	Speed 280.9 samples/s	Data 0.000s (0.004s)	Loss 0.17408 (0.18244)
Epoch: [16][60/4375]	Time 0.057s (0.059s)	Speed 279.4 samples/s	Data 0.000s (0.003s)	Loss 0.17520 (0.18270)
INFO:train:Epoch: [16][60/4375]	Time 0.057s (0.059s)	Speed 279.4 samples/s	Data 0.000s (0.003s)	Loss 0.17520 (0.18270)
Epoch: [16][80/4375]	Time 0.0

                 all       1e+04    1.08e+05      0.0498       0.873       0.699       0.368
Speed: 1.3/0.4/1.7 ms inference/NMS/total per 640x640 image at batch-size 16
Results saved to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolop/visualization


  ** new best_det: mAP50=0.6991
INFO:train:  ** new best_det: mAP50=0.6991
  ** new best_lane: LL_IoU=0.2063
INFO:train:  ** new best_lane: LL_IoU=0.2063
  ** new best_joint: 0.4527
INFO:train:  ** new best_joint: 0.4527
Epoch: [17][0/4375]	Time 0.261s (0.261s)	Speed 61.4 samples/s	Data 0.210s (0.210s)	Loss 0.17429 (0.17429)
INFO:train:Epoch: [17][0/4375]	Time 0.261s (0.261s)	Speed 61.4 samples/s	Data 0.210s (0.210s)	Loss 0.17429 (0.17429)
Epoch: [17][20/4375]	Time 0.058s (0.067s)	Speed 277.9 samples/s	Data 0.000s (0.010s)	Loss 0.17834 (0.18142)
INFO:train:Epoch: [17][20/4375]	Time 0.058s (0.067s)	Speed 277.9 samples/s	Data 0.000s (0.010s)	Loss 0.17834 (0.18142)
Epoch: [17][40/4375]	Time 0.056s (0.062s)	Speed 285.9 samples/s	Data 0.000s (0.005s)	Loss 0.16583 (0.18169)
INFO:train:Epoch: [17][40/4375]	Time 0.056s (0.062s)	Speed 285.9 samples/s	Data 0.000s (0.005s)	Loss 0.16583 (0.18169)
Epoch: [17][60/4375]	Time 0.053s (0.060s)	Speed 303.2 samples/s	Data 0.000s (0.004s)	Loss 0.17578 (0.1

                 all       1e+04    1.08e+05      0.0448       0.871       0.696       0.363
Speed: 1.3/0.4/1.7 ms inference/NMS/total per 640x640 image at batch-size 16
Results saved to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolop/visualization


Epoch: [18][0/4375]	Time 0.250s (0.250s)	Speed 64.0 samples/s	Data 0.184s (0.184s)	Loss 0.17190 (0.17190)
INFO:train:Epoch: [18][0/4375]	Time 0.250s (0.250s)	Speed 64.0 samples/s	Data 0.184s (0.184s)	Loss 0.17190 (0.17190)
Epoch: [18][20/4375]	Time 0.049s (0.065s)	Speed 323.8 samples/s	Data 0.000s (0.009s)	Loss 0.18598 (0.18208)
INFO:train:Epoch: [18][20/4375]	Time 0.049s (0.065s)	Speed 323.8 samples/s	Data 0.000s (0.009s)	Loss 0.18598 (0.18208)
Epoch: [18][40/4375]	Time 0.049s (0.060s)	Speed 323.9 samples/s	Data 0.000s (0.005s)	Loss 0.18597 (0.17996)
INFO:train:Epoch: [18][40/4375]	Time 0.049s (0.060s)	Speed 323.9 samples/s	Data 0.000s (0.005s)	Loss 0.18597 (0.17996)
Epoch: [18][60/4375]	Time 0.050s (0.058s)	Speed 320.0 samples/s	Data 0.000s (0.003s)	Loss 0.18713 (0.18065)
INFO:train:Epoch: [18][60/4375]	Time 0.050s (0.058s)	Speed 320.0 samples/s	Data 0.000s (0.003s)	Loss 0.18713 (0.18065)
Epoch: [18][80/4375]	Time 0.055s (0.057s)	Speed 290.1 samples/s	Data 0.000s (0.002s)	Loss 0.1836